# Assignment — Linear Regression on `penguins.csv`

**Question:** given a penguin's measurements, how heavy is it?

Target: `body_mass_g` &nbsp;·&nbsp; File: `penguins.csv`

Use only the numeric columns: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`.

In [2]:
import numpy as np
import pandas as pd
df = pd.read_csv('penguins.csv')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


---
## 1. Look at the data

> **Flow:** Shape, columns, missing values.

Run `.info()`. Which columns have missing values, and how many?

In [ ]:
df.info()
print("Shape:", df.shape)
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB
Shape: (344, 7)
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


---
## 2. Clean

> **Flow:** Keep the four numeric columns, drop the rows with blanks.

We cannot fill in `body_mass_g` — that is the answer we are trying to predict.
Rows missing it have to go.

Keep `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`,
then `dropna()`.

In [4]:

data = df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']].dropna()
print(data.shape)

(342, 4)


*How many rows are left?*

342 (We remove rows with missing body_mass_g because it is the target we are trying to predict.)

---
## 3. Features and target

In [5]:
X = data[['bill_length_mm','bill_depth_mm','flipper_length_mm']]
y = data['body_mass_g']

---
## 4. Split

> **Flow:** `test_size=0.2`, `random_state=42`.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

---
## 5. One feature

> **Flow:** Start with `flipper_length_mm` alone.

Train `LinearRegression`. Print the coefficient, the intercept and the R².

In [7]:
from sklearn.linear_model import LinearRegression

X1 = data[['flipper_length_mm']]

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y,
    test_size=0.2,
    random_state=42
)

model1 = LinearRegression()
model1.fit(X1_train, y1_train)

y1_pred = model1.predict(X1_test)

print("Coefficient:", model1.coef_[0])
print("Intercept:", model1.intercept_)
print("R²:", model1.score(X1_test, y1_test))

Coefficient: 48.82968281899302
Intercept: -5614.067120606903
R²: 0.7820354165340793


*Coefficient:* &nbsp;&nbsp; *R²:*

**Q.** The coefficient is about 48. Write one line explaining what that means in plain words —
what happens to the predicted weight if a penguin's flipper is 1 mm longer?

For every 1 mm increase in flipper length, the predicted body mass increases by approximately 48.8 g.

---
## 6. All three features

> **Flow:** Same model, two more columns.

Print the R².

In [8]:
X = data[['bill_length_mm','bill_depth_mm','flipper_length_mm']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R²:", model.score(X_test, y_test))

R²: 0.7877806019338432


*R² with flipper only:  0.7820* &nbsp;&nbsp; *R² with all three: 0.7878*

**Q.** The score barely moved. That is not a mistake — it is the interesting part of this
assignment. Write two lines on why adding `bill_length_mm` and `bill_depth_mm` gave almost
nothing.

*Hint: what do all three columns really measure?*

Ans: bill_length_mm, bill_depth_mm, and flipper_length_mm are all physical measurements of the penguin's body. They contain related information about body size, so flipper length already captures most of the useful information for predicting body mass.

Therefore, adding bill measurements provides only a small additional improvement.

---
## 8. Metrics

> **Flow:** MAE, RMSE, R².

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 310.54744049126754
RMSE: 375.6441343110777
R²: 0.7877806019338432


*MAE: 310.55 g* &nbsp;&nbsp; *RMSE: 375.64 g* &nbsp;&nbsp; *R²:  0.7878*

**Q.** The average penguin weighs about 4200 g. Is your MAE large or small compared to that?
One line.

Ans: Average penguin weight is about 4200 g.
So an MAE of 310.55 g is relatively small — the predictions are off by about 311 g on average, or roughly 7.4% of the average body mass.

---
## 8. Coefficients

In [10]:
# Print each feature name next to its coefficient
for feature, coefficient in zip(X.columns, model.coef_):
    print(feature, ":", coefficient)

bill_length_mm : 4.00878845592763
bill_depth_mm : 10.924559578456657
flipper_length_mm : 48.67371867581057


*Largest coefficient:*

**Q.** Does the largest coefficient mean that feature is the most important?
Careful — the three columns are measured on different scales. One line.

Ans: No. A larger coefficient does not necessarily mean greater importance because the features are measured on different scales. The coefficient represents the change in predicted weight for a 1-unit change in that particular feature.

---
## 9. Questions

**Q1.** Why can we not use `accuracy_score` on this problem?

Ans: Because this is a regression problem, where body_mass_g is a continuous numerical value. accuracy_score is designed for classification, where predictions are discrete classes.

**Q2.** We dropped rows where `body_mass_g` was missing instead of filling them with the
median. Why is filling in the target a bad idea?

Ans: Because body_mass_g is the target we are trying to predict. Filling missing target values with the median would create artificial labels and give the model incorrect information.

**Q3.** In class, adding more features to the mpg model raised R² from 0.723 to 0.824.
Here it barely changed. In two lines — what is different about these two datasets?

Ans: For penguins, the three measurements are all related to physical body size, and flipper_length_mm already explains most of the variation in body mass. Therefore, adding bill length and bill depth provides little extra information.